# Строгий датасет связанных объектов



В результат попадают только объекты недвижимости, для которых подтверждена цепочка:

`договор → заявка → выбранная задача → связь с объектом → характеристики → условия`.



In [ ]:
%pip install pandas sqlalchemy "psycopg[binary]" oracledb

In [ ]:
import json
from pathlib import Path
import oracledb
import pandas as pd
from sqlalchemy import URL, create_engine, text

pd.set_option('display.max_columns', 100)

In [ ]:
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / 'уч данные.txt').exists():
    NOTEBOOK_DIR = CURRENT_DIR
elif (CURRENT_DIR / 'notebooks' / 'уч данные.txt').exists():
    NOTEBOOK_DIR = CURRENT_DIR / 'notebooks'
else:
    NOTEBOOK_DIR = CURRENT_DIR

PROJECT_ROOT = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name == 'notebooks'
    else NOTEBOOK_DIR
)
OUTPUT_DIR = PROJECT_ROOT / 'РЕЗУЛЬТАТЫ_ЛОКАЛЬНО'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Корень проекта:', PROJECT_ROOT)
print('Папка результатов:', OUTPUT_DIR)

# 1. Подключение к Сфере


In [ ]:
CREDENTIALS_PATH = NOTEBOOK_DIR / 'уч данные.txt'

def read_credentials(path):
    if not path.exists():
        raise FileNotFoundError(f'Не найден файл с учётными данными: {path}')

    credentials = {}
    for line_number, raw_line in enumerate(
        path.read_text(encoding='utf-8-sig').splitlines(),
        start=1,
    ):
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if '=' not in line:
            raise ValueError(
                f'Строка {line_number}: ожидается запись КЛЮЧ=значение'
            )

        key, value = line.split('=', 1)
        credentials[key.strip()] = value.strip()

    return credentials

credentials = read_credentials(CREDENTIALS_PATH)

sphere_required = [
    'SPHERE_HOST',
    'SPHERE_DATABASE',
    'SPHERE_USER',
    'SPHERE_PASSWORD',
]
sphere_missing = [key for key in sphere_required if not credentials.get(key)]
if sphere_missing:
    raise ValueError(
        'Заполни в уч данные.txt: ' + ', '.join(sphere_missing)
    )

SPHERE_HOST = credentials['SPHERE_HOST']
SPHERE_PORT = int(credentials.get('SPHERE_PORT', '5432'))
SPHERE_DATABASE = credentials['SPHERE_DATABASE']
SPHERE_USER = credentials['SPHERE_USER']
SPHERE_PASSWORD = credentials['SPHERE_PASSWORD']

connection_url = URL.create(
    drivername='postgresql+psycopg',
    username=SPHERE_USER,
    password=SPHERE_PASSWORD,
    host=SPHERE_HOST,
    port=SPHERE_PORT,
    database=SPHERE_DATABASE,
)
engine = create_engine(connection_url, pool_pre_ping=True)

print('Учётные данные прочитаны, подключение к Сфере создано')


In [ ]:
with engine.connect() as connection:
    connection_check = pd.read_sql_query(
        text('select current_database() as database_name, current_user as user_name'),
        connection,
    )

display(connection_check)

# 2. Подключение к Oracle КХД



In [ ]:
khd_required = [
    'KHD_HOST',
    'KHD_SERVICE_NAME',
    'KHD_USER',
    'KHD_PASSWORD',
]
khd_missing = [key for key in khd_required if not credentials.get(key)]
if khd_missing:
    raise ValueError(
        'Заполни в уч данные.txt: ' + ', '.join(khd_missing)
    )

KHD_HOST = credentials['KHD_HOST']
KHD_PORT = int(credentials.get('KHD_PORT', '1521'))
KHD_SERVICE_NAME = credentials['KHD_SERVICE_NAME']
KHD_USER = credentials['KHD_USER']
KHD_PASSWORD = credentials['KHD_PASSWORD']
KHD_DATA_SCHEMA = credentials.get('KHD_DATA_SCHEMA', 'DM_RISK_AVATAR')

khd_dsn = oracledb.makedsn(
    KHD_HOST,
    KHD_PORT,
    service_name=KHD_SERVICE_NAME,
)
khd_connection = oracledb.connect(
    user=KHD_USER,
    password=KHD_PASSWORD,
    dsn=khd_dsn,
)

print('Подключение к КХД создано')


In [ ]:
with khd_connection.cursor() as cursor:
    cursor.execute(
        "select user, sys_context('USERENV', 'DB_NAME') from dual"
    )
    khd_user_name, khd_database_name = cursor.fetchone()

    cursor.execute(
        """
        select table_name
        from all_tables
        where owner = :owner
          and table_name = 'EGRN_DATA'
        order by table_name
        """,
        owner=KHD_DATA_SCHEMA.upper(),
    )
    available_khd_tables = [row[0] for row in cursor.fetchall()]

print('Пользователь КХД:', khd_user_name)
print('База КХД:', khd_database_name)
print('Доступные таблицы:', available_khd_tables)

expected_khd_tables = {'EGRN_DATA'}
missing_khd_tables = sorted(expected_khd_tables - set(available_khd_tables))
if missing_khd_tables:
    raise PermissionError(
        'Не видны таблицы КХД: ' + ', '.join(missing_khd_tables)
    )

# 3. SQL Сфера, строгий подход

In [ ]:
strict_sql = r"""
/*
Для чего нужен запрос
---------------------
Запрос собирает основу датасета для модели 1 по недвижимости ЮЛ.
Он объединяет сведения о договоре, объекте, адресе, страхователе,
отрасли, страховых суммах и ближайшем предыдущем договоре.

Одна строка результата
----------------------
Одна строка - один объект недвижимости в одном договоре.
Один договор может занимать несколько строк, если в нем несколько объектов.


Как связаны таблицы
-------------------
Договор -> заявка -> задача оформления -> объект в задаче
        -> характеристики объекта -> сам объект -> адрес
        -> условия страхования объекта

Из договора берется страхователь. Из заявки берется CRM-карточка,
в которой находятся отрасль и сегмент.

Какие записи попадают в результат
---------------------------------
- задача оформления договора: task_type = draft_contract;
- завершенная рабочая задача: status = operational_archive;
- тип документа: ins_document_type = new_ins_contract,
  ins_contract_prolong или NULL, то есть новый договор, пролонгация
  или незаполненное значение;
- отказ в страховании не установлен: ins_refuse IS NOT TRUE;
- дата удаления отсутствует: d_delete IS NULL для задачи, заявки,
  договора и объекта;
- тип объекта: elementary_obj_type = nedv_ul_and_ip,
  то есть недвижимость ЮЛ и ИП.


Как читать страховые суммы
--------------------------
- contract_insured_sum - общая СС всего договора;
- task_object_insured_sum - СС объекта в строке связи задачи и объекта;
- condition_min_insured_sum и condition_max_insured_sum - минимальная и
  максимальная СС среди условий выбранной версии объекта;
- insured_sum - СС среди условий выбранной версии объекта.


Как используется история
-------------------------
Ближайший предыдущий договор ищется по bps_contract.prevcontract_id.
Прошлая СС объекта заполняется только тогда, когда в текущем и предыдущем
договоре совпал object_id. Если при пролонгации объект завели с новым ID,
прошлая объектная СС останется пустой.

*/

with task_candidates as (
    /* Шаг 1. Находим все подходящие задачи оформления. */
    select
        c.id as contract_id,
        c.n_contract as contract_number,
        c.prevcontract_id as previous_contract_id,
        c.rootcontract_id as root_contract_id,
        c.contractor_id as policyholder_id,
        c.document_status as contract_status,
        c.d_sign_contract as contract_sign_date,
        c.d_start_contract as contract_start_date,
        c.d_end_contract as contract_end_date,
        c.currency as contract_currency,
        c.ins_product_sbs as insurance_product,
        c.ins_program as insurance_program,

        r.id as request_id,
        r.corporate_crm_id,
        r.business_segment,

        t.id as task_id,
        t.d_create as task_create_date,
        t.d_change as task_change_date,
        t.task_type,
        t.status as task_status,
        t.ins_document_type,
        t.contract_type,
        t.d_conclusion_ins_contract as contract_conclusion_date,
        t.ins_refuse,
        t.industry as task_industry,
        t.subindustry as task_subindustry,
        t.total_ins_contract_amount as contract_insured_sum,
        t.total_ins_contract_premium as contract_premium,
        t.curr_ins_contract_amount as contract_amount_currency,

        coalesce(
            t.d_conclusion_ins_contract::timestamp with time zone,
            c.d_sign_contract,
            t.d_create
        ) as as_of_date,

        row_number() over (
            partition by c.id
            order by
                coalesce(
                    t.d_conclusion_ins_contract::timestamp with time zone,
                    t.d_create,
                    t.d_change
                ) desc nulls last,
                t.d_create desc nulls last,
                t.d_change desc nulls last,
                t.id desc
        ) as task_number
    from bps_request_ins_task t
    join bps_request_ins r
        on r.id = t.request_ins_id
    join bps_contract c
        on c.id = r.contract_id
    where t.task_type = 'draft_contract'
      and t.status = 'operational_archive'
      and (
          t.ins_document_type = 'new_ins_contract'
          or t.ins_document_type = 'ins_contract_prolong'
          or t.ins_document_type is null
      )
      and t.ins_refuse is not true
      and t.d_delete is null
      and r.d_delete is null
      and c.d_delete is null
),

selected_tasks as (
    /* Шаг 2. Для каждого договора оставляем одну самую позднюю задачу. */
    select *
    from task_candidates
    where task_number = 1
),

contract_context as (
    /* Шаг 3. Добавляем ближайший предыдущий договор, если он указан. */
    select
        current_task.*,
        previous_contract.n_contract as previous_contract_number,
        previous_contract.d_start_contract as previous_contract_start_date,
        previous_contract.d_end_contract as previous_contract_end_date,
        previous_task.contract_insured_sum as previous_contract_insured_sum,
        previous_task.contract_premium as previous_contract_premium,
        previous_task.contract_amount_currency
            as previous_contract_amount_currency
    from selected_tasks current_task
    left join bps_contract previous_contract
        on previous_contract.id = current_task.previous_contract_id
    left join selected_tasks previous_task
        on previous_task.contract_id = current_task.previous_contract_id
),

object_candidates as (
    /*
    Шаг 4. К выбранной задаче присоединяем объекты недвижимости.
    Здесь же добавляем адрес, страхователя, CRM и характеристики объекта.
    */
    select
        contract.*,

        policyholder.inn as policyholder_inn,
        policyholder.contractor_type as policyholder_type,
        policyholder.cdi_id as policyholder_cdi_id,
        policyholder.ogrn as policyholder_ogrn,
        policyholder.kpp as policyholder_kpp,
        policyholder.company_name_short as policyholder_name,
        policyholder.company_form as policyholder_company_form,
        policyholder.company_register_day
            as policyholder_registration_date,

        crm.id as crm_id,
        crm.client_id as crm_client_id,
        crm.segment as crm_segment,
        crm.macroindustry as crm_macroindustry,
        crm.industry as crm_industry,
        crm.primary_occupation as crm_primary_occupation,
        crm.specialization as crm_specialization,
        crm.okved as crm_okved,

        link.id as task_object_link_id,
        link.characteristics_id,
        link.object_group_id,
        link.insured_sum as task_object_insured_sum,
        link.insured_sum_currency as task_object_insured_sum_currency,
        link.per_occurance_limit as task_object_per_occurrence_limit,

        obj.id as object_id,
        obj.obj_name as object_name,
        obj.description as object_description,
        obj.obj_type as object_type,
        obj.elementary_obj_type,
        obj.original_address,
        obj.geo_address_id,

        ch.version_number as characteristics_version_number,
        ch.version_start_date as characteristics_version_start_date,
        ch.version_end_date as characteristics_version_end_date,
        ch.version_is_active as characteristics_version_is_active,
        ch.insurance_value,
        ch.insurance_value_currency,
        ch.insurance_value_basis,
        ch.is_pledged,
        ch.pledged_value,
        ch.ownership_type,
        ch.is_leased,
        ch.insured_components,
        ch.activity_types,
        ch.risk_natures,
        ch.insurance_territory,
        ch.characteristics ->> 'total_area_sq_m' as total_area,
        ch.characteristics ->> 'occupied_area_sq_m' as occupied_area,
        ch.characteristics ->> 'construction_year' as construction_year,
        ch.characteristics ->> 'last_capital_repair_year'
            as capital_repair_year,
        ch.characteristics ->> 'total_floors_count' as floors_count,
        ch.characteristics ->> 'occupied_floor' as occupied_floor,
        ch.characteristics ->> 'load_bearing_walls_material'
            as walls_material,
        ch.characteristics ->> 'interfloor_overlap_material'
            as overlap_material,
        ch.characteristics ->> 'roofing_material' as roofing_material,
        ch.characteristics as object_characteristics_json,

        address.full_address,
        address.postal_code,
        address.region_id as address_region_id,
        address.area as district,
        address.settlement,
        address.street,
        address.house,
        address.building,
        address.block,
        address.flat,
        address.office,
        address.fias_code,
        address.longitude,
        address.latitude,
        address.address_dgis_id,

        row_number() over (
            partition by contract.task_id, obj.id
            order by
                link.d_change desc nulls last,
                link.d_create desc nulls last,
                ch.version_start_date desc nulls last,
                ch.version_number desc nulls last,
                link.id desc,
                ch.id desc
        ) as object_number
    from contract_context contract
    join bps_request_ins_task_insurance_object link
        on link.parent_id = contract.task_id
    join base_insurance_object_characteristics ch
        on ch.id = link.characteristics_id
    join base_insurance_object obj
        on obj.id = ch.insurance_object_id
    left join base_geo_address address
        on address.id = obj.geo_address_id
    left join bps_contractor policyholder
        on policyholder.id = contract.policyholder_id
    left join bps_corporate_crm crm
        on crm.id = contract.corporate_crm_id
    where obj.elementary_obj_type = 'nedv_ul_and_ip'
      and obj.d_delete is null
),

selected_objects as (
    /*
    Шаг 5. Если объект несколько раз связан с одной задачей,
    оставляем одну самую позднюю запись связи.
    */
    select *
    from object_candidates
    where object_number = 1
),

selected_characteristics as (
    /* Шаг 6. Получаем список версий объектов для поиска их условий. */
    select distinct characteristics_id
    from selected_objects
),

condition_summary as (
    /*
    Шаг 7. У одной версии объекта может быть несколько вариантов условий.
    Сворачиваем их в одну строку, чтобы один объект не продублировался.
    */
    select
        cond.characteristics_id,
        count(*) as condition_count,
        min(cond.insured_sum) as condition_min_insured_sum,
        max(cond.insured_sum) as condition_max_insured_sum,
        count(distinct cond.insured_sum_currency) filter (
            where cond.insured_sum_currency is not null
        ) as condition_currency_count,
        string_agg(
            distinct cond.insured_sum_currency,
            ', '
            order by cond.insured_sum_currency
        ) filter (
            where cond.insured_sum_currency is not null
        ) as insured_sum_currency,
        min(cond.per_occurance_limit) as minimum_per_occurrence_limit,
        max(cond.per_occurance_limit) as maximum_per_occurrence_limit,
        jsonb_agg(
            jsonb_strip_nulls(
                jsonb_build_object(
                    'option_number', cond.terms_option_number,
                    'insured_sum', cond.insured_sum,
                    'currency', cond.insured_sum_currency,
                    'per_occurrence_limit', cond.per_occurance_limit
                )
            )
            order by cond.terms_option_number nulls last, cond.id
        ) as conditions_json
    from base_insurance_object_conditions cond
    join selected_characteristics selected
        on selected.characteristics_id = cond.characteristics_id
    group by cond.characteristics_id
),

object_data as (
    /* Шаг 8. Добавляем к каждому объекту найденные суммы и условия. */
    select
        obj.*,
        conditions.condition_count,
        conditions.condition_min_insured_sum,
        conditions.condition_max_insured_sum,
        conditions.condition_currency_count,
        conditions.insured_sum_currency,
        conditions.minimum_per_occurrence_limit,
        conditions.maximum_per_occurrence_limit,
        conditions.conditions_json
    from selected_objects obj
    left join condition_summary conditions
        on conditions.characteristics_id = obj.characteristics_id
),

objects_with_previous as (
    /*
    Шаг 9. Ищем тот же object_id в ближайшем предыдущем договоре
    и, если нашли, добавляем его предыдущую СС.
    */
    select
        current_object.*,
        case
            when previous_object.condition_min_insured_sum =
                 previous_object.condition_max_insured_sum
             and previous_object.condition_currency_count <= 1
            then previous_object.condition_max_insured_sum
        end as previous_object_insured_sum,
        previous_object.insured_sum_currency
            as previous_object_insured_sum_currency
    from object_data current_object
    left join object_data previous_object
        on previous_object.contract_id = current_object.previous_contract_id
       and previous_object.object_id = current_object.object_id
),

raw_result as (
/* Шаг 10. Собираем исходные поля строгого датасета. */
select
    /* Основные ID. */
    obj.contract_id,
    obj.contract_number,
    obj.previous_contract_id,
    obj.root_contract_id,
    obj.request_id,
    obj.task_id,
    obj.task_object_link_id,
    obj.characteristics_id,
    obj.object_id,
    obj.geo_address_id,
    obj.policyholder_id,
    obj.corporate_crm_id,

    /* Договор и его даты. */
    obj.as_of_date,
    obj.contract_conclusion_date,
    obj.contract_sign_date,
    obj.contract_start_date,
    obj.contract_end_date,
    obj.contract_status,
    obj.ins_document_type,
    obj.contract_type,
    obj.contract_currency,
    obj.insurance_product,
    obj.insurance_program,

    /* Объект. */
    count(*) over (
        partition by obj.contract_id
    ) as real_estate_objects_in_contract,
    obj.object_group_id,
    obj.object_name,
    obj.object_description,
    obj.object_type,
    obj.elementary_obj_type,
    obj.total_area,
    obj.occupied_area,
    obj.construction_year,
    obj.capital_repair_year,
    obj.floors_count,
    obj.occupied_floor,
    obj.walls_material,
    obj.overlap_material,
    obj.roofing_material,
    obj.ownership_type,
    obj.is_leased,
    obj.insured_components,
    obj.activity_types,
    obj.risk_natures,
    obj.insurance_territory,

    /* Адрес. */
    obj.full_address,
    obj.original_address,
    obj.postal_code,
    obj.address_region_id,
    obj.district,
    obj.settlement,
    obj.street,
    obj.house,
    obj.building,
    obj.block,
    obj.flat,
    obj.office,
    obj.fias_code,
    obj.longitude,
    obj.latitude,
    obj.address_dgis_id,

    /* Страхователь, отрасль и сегмент. */
    obj.policyholder_inn,
    obj.policyholder_type,
    obj.policyholder_cdi_id,
    obj.policyholder_ogrn,
    obj.policyholder_kpp,
    obj.policyholder_name,
    obj.policyholder_company_form,
    obj.policyholder_registration_date,
    obj.crm_id,
    obj.crm_client_id,
    (obj.crm_client_id = obj.policyholder_id) as crm_client_is_policyholder,
    obj.crm_segment,
    obj.crm_macroindustry,
    obj.crm_industry,
    obj.crm_primary_occupation,
    obj.crm_specialization,
    obj.crm_okved,
    obj.business_segment,
    obj.task_industry,
    obj.task_subindustry,

    /*
    Все СС стоят рядом.
    СС договора относится ко всему договору и повторяется у его объектов.
    */
    obj.contract_insured_sum,
    obj.contract_amount_currency,
    min(obj.condition_min_insured_sum) over (
        partition by obj.contract_id
    ) as contract_real_estate_min_insured_sum,
    max(obj.condition_max_insured_sum) over (
        partition by obj.contract_id
    ) as contract_real_estate_max_insured_sum,
    obj.task_object_insured_sum,
    obj.task_object_insured_sum_currency,
    obj.condition_min_insured_sum,
    obj.condition_max_insured_sum,
    case
        /* Не выбираем случайную СС, если в условиях есть расхождения. */
        when obj.condition_min_insured_sum =
             obj.condition_max_insured_sum
         and obj.condition_currency_count <= 1
        then obj.condition_max_insured_sum
    end as insured_sum,
    obj.insured_sum_currency,
    obj.condition_currency_count,
    obj.previous_contract_insured_sum,
    obj.previous_contract_amount_currency,
    obj.previous_object_insured_sum,
    obj.previous_object_insured_sum_currency,

    /* Премии, стоимости и лимиты. */
    obj.contract_premium,
    obj.previous_contract_premium,
    obj.insurance_value,
    obj.insurance_value_currency,
    obj.insurance_value_basis,
    obj.is_pledged,
    obj.pledged_value,
    obj.task_object_per_occurrence_limit,
    obj.minimum_per_occurrence_limit,
    obj.maximum_per_occurrence_limit,

    /* Простая договорная история. */
    (obj.previous_contract_id is not null) as has_previous_contract,
    obj.previous_contract_number,
    obj.previous_contract_start_date,
    obj.previous_contract_end_date,

    /* Исходные данные для проверки. */
    obj.condition_count,
    obj.conditions_json,
    obj.characteristics_version_number,
    obj.characteristics_version_start_date,
    obj.characteristics_version_end_date,
    obj.characteristics_version_is_active,
    obj.object_characteristics_json,
    obj.task_type,
    obj.task_status,
    obj.ins_refuse
from objects_with_previous obj
),

standardized_result as (
    /* Шаг 11. Приводим результат к общей структуре двух датасетов. */
    select
        case
            when raw.contract_id is null then 'not_linked'
            else 'linked'
        end as row_source,
        case
            when count(raw.contract_id) over (
                partition by raw.object_id
            ) = 0 then 'not_linked'
            when count(raw.contract_id) over (
                partition by raw.object_id
            ) = 1 then 'linked'
            else 'multiple_contracts'
        end as contract_link_status,
        count(raw.contract_id) over (
            partition by raw.object_id
        ) as contract_count,
        (raw.contract_id is not null) as has_contract,
        (
            raw.geo_address_id is not null
            or nullif(btrim(raw.full_address), '') is not null
            or nullif(btrim(raw.original_address), '') is not null
        ) as has_address,
        (raw.insured_sum is not null) as has_target,
        case
            when raw.condition_count is null
              or raw.condition_count = 0
                then 'no_conditions'
            when raw.condition_min_insured_sum is distinct from
                 raw.condition_max_insured_sum
                then 'several_target_values'
            when coalesce(raw.condition_currency_count, 0) > 1
                then 'several_currencies'
            when raw.insured_sum <= 0
                then 'target_is_not_positive'
            when raw.insured_sum is null
                then 'target_is_empty'
            else 'target_is_usable'
        end as target_status,

        raw.contract_id,
        raw.contract_number,
        raw.previous_contract_id,
        raw.root_contract_id,
        raw.request_id,
        raw.task_id,
        raw.task_object_link_id,
        raw.characteristics_id,
        raw.object_id,
        raw.geo_address_id,
        raw.policyholder_id,
        raw.corporate_crm_id,

        raw.as_of_date,
        raw.contract_conclusion_date,
        raw.contract_sign_date,
        raw.contract_start_date,
        raw.contract_end_date,
        raw.contract_status,
        raw.ins_document_type,
        raw.insurance_product,

        case
            when raw.contract_id is not null then
                count(raw.object_id) over (
                    partition by raw.contract_id
                )
        end as real_estate_objects_in_contract,
        raw.object_name,
        raw.object_description,
        raw.object_type,
        raw.elementary_obj_type,
        raw.total_area,
        raw.occupied_area,
        raw.construction_year,
        raw.capital_repair_year,
        raw.floors_count,
        raw.occupied_floor,
        raw.walls_material,
        raw.overlap_material,
        raw.roofing_material,
        raw.ownership_type,
        raw.is_leased,
        raw.insured_components,
        raw.activity_types,
        raw.risk_natures,
        raw.insurance_territory,

        raw.full_address,
        raw.original_address,
        raw.postal_code,
        raw.address_region_id,
        raw.district,
        raw.settlement,
        raw.street,
        raw.house,
        raw.building,
        raw.block,
        raw.flat,
        raw.office,
        raw.fias_code,
        raw.longitude,
        raw.latitude,
        raw.address_dgis_id,

        raw.policyholder_inn,
        raw.policyholder_name,
        raw.policyholder_cdi_id,
        raw.crm_segment,
        raw.crm_macroindustry,
        raw.crm_industry,
        raw.crm_okved,
        raw.business_segment,
        raw.task_industry,
        raw.task_subindustry,

        raw.contract_insured_sum,
        raw.contract_amount_currency,
        raw.task_object_insured_sum,
        raw.task_object_insured_sum_currency,
        raw.condition_min_insured_sum,
        raw.condition_max_insured_sum,
        raw.insured_sum,
        raw.insured_sum_currency,
        raw.condition_currency_count,
        raw.contract_premium,
        raw.insurance_value,
        raw.insurance_value_currency,
        raw.insurance_value_basis,
        raw.is_pledged,
        raw.pledged_value,
        raw.minimum_per_occurrence_limit,
        raw.maximum_per_occurrence_limit,

        raw.previous_contract_number,
        raw.previous_contract_start_date,
        raw.previous_contract_end_date,

        raw.condition_count,
        raw.characteristics_version_number,
        raw.characteristics_version_start_date,
        raw.characteristics_version_end_date,
        raw.characteristics_version_is_active,
        raw.object_characteristics_json,
        raw.task_type,
        raw.task_status,
        raw.ins_refuse
    from raw_result raw
)

select *
from standardized_result
order by
    as_of_date desc nulls last,
    contract_id,
    object_id;

"""


In [ ]:
with engine.connect() as connection:
    strict_df = pd.read_sql_query(text(strict_sql), connection)

print('Строк:', len(strict_df))
print('Колонок:', len(strict_df.columns))
display(strict_df.head(3))


# 4. Проверка заполненности

In [ ]:
required_columns = {
    'contract_id', 'task_id', 'object_id', 'characteristics_id',
    'elementary_obj_type', 'insured_sum', 'full_address'
}
missing_columns = sorted(required_columns - set(strict_df.columns))
if missing_columns:
    raise ValueError('Не найдены ожидаемые колонки: ' + ', '.join(missing_columns))

profile = pd.DataFrame({
    'Показатель': [
        'Строк',
        'Уникальных договоров',
        'Уникальных задач',
        'Уникальных объектов',
        'Уникальных пар задача + объект',
        'Строк с target',
        'Строк с адресом',
        'Строк с пустым типом объекта',
    ],
    'Значение': [
        len(strict_df),
        strict_df['contract_id'].nunique(dropna=True),
        strict_df['task_id'].nunique(dropna=True),
        strict_df['object_id'].nunique(dropna=True),
        strict_df[['task_id', 'object_id']].drop_duplicates().shape[0],
        strict_df['insured_sum'].notna().sum(),
        strict_df['full_address'].fillna('').str.strip().ne('').sum(),
        strict_df['elementary_obj_type'].fillna('').str.strip().eq('').sum(),
    ],
})

display(profile)

In [ ]:
duplicate_keys = (
    strict_df.groupby(['task_id', 'object_id'], dropna=False)
    .size()
    .gt(1)
    .sum()
)
print('Повторных ключей задача + объект:', duplicate_keys)
display(strict_df['elementary_obj_type'].fillna('empty').value_counts(dropna=False))

# 5. Соединение с ЕГРН

Объекты передаются в один Oracle `SELECT` как JSON-параметр. В КХД ничего не создаётся и не записывается. Поиск выполняется на уровне здания. Данные ЕГРН присоединяются только при одном кандидате; неоднозначные случаи остаются пустыми.


In [ ]:
# готовим только те поля, которые нужны Oracle для поиска ЕГРН
sphere_with_row_id = strict_df.copy()
sphere_with_row_id.insert(0, 'sphere_row_id', range(1, len(sphere_with_row_id) + 1))

stage_columns = [
    'sphere_row_id',
    'contract_id',
    'contract_number',
    'task_id',
    'task_object_link_id',
    'characteristics_id',
    'object_id',
    'geo_address_id',
    'real_estate_objects_in_contract',
    'object_description',
    'total_area',
    'full_address',
    'original_address',
    'postal_code',
    'settlement',
    'street',
    'house',
    'building',
    'block',
    'flat',
    'office',
]

missing_stage_columns = [
    column for column in stage_columns
    if column not in sphere_with_row_id.columns
]
if missing_stage_columns:
    raise ValueError(
        'Для поиска ЕГРН не хватает колонок: '
        + ', '.join(missing_stage_columns)
    )

def json_value(column, value):
    if value is None or pd.isna(value):
        return None
    if column == 'sphere_row_id':
        return int(value)
    return str(value)

sphere_records = []
for row in sphere_with_row_id[stage_columns].itertuples(index=False, name=None):
    sphere_records.append({
        column: json_value(column, value)
        for column, value in zip(stage_columns, row)
    })

sphere_json = json.dumps(sphere_records, ensure_ascii=False)
print('Строк передано в Oracle SELECT:', len(sphere_records))
print('Размер JSON, МБ:', round(len(sphere_json.encode('utf-8')) / 1024**2, 2))


In [ ]:
egrn_sql = r"""
/*
Соединение временной выборки объектов Сферы с ЕГРН по адресу.

Запускать в Oracle.

Результат содержит одну строку на объект Сферы. Данные ЕГРН заполняются
только тогда, когда по адресу найдена одна запись уровня здания.

Для поиска используются населённый пункт, улица и дом. Корпус и строение
учитываются, если они указаны.
Индекс и регион сравниваются, если они заполнены с обеих сторон.
Если по адресу найдено несколько зданий, площадь используется как
дополнительная проверка. Если площади нет или она не помогла, кандидаты
по адресу не отсекаются.

В поиск попадают только типы ЕГРН «здание», «сооружение» и «строение»
с уровнем адреса FIAS_HOUSE. Квартиры, офисы, комнаты и помещения исключены.
Если по одному адресу Сферы записано несколько страховых объектов,
одно найденное здание ЕГРН присоединяется к каждому из них.

Источник передаётся из notebook одним JSON-параметром `sphere_json`.
Запрос только читает EGRN_DATA и ничего не создаёт в Oracle.
*/

with sphere_source as (
    /* Берём только поля, которые нужны для проверки соединения. */
    select /*+ materialize */
        s.sphere_row_id,
        s.contract_id,
        s.contract_number,
        s.task_id,
        s.task_object_link_id,
        s.characteristics_id,
        s.object_id,
        s.geo_address_id,
        s.real_estate_objects_in_contract,
        cast(s.object_description as varchar2(4000))
            as object_description,
        s.total_area,
        cast(s.full_address as varchar2(4000)) as full_address,
        cast(s.original_address as varchar2(4000)) as original_address,
        s.postal_code,
        s.settlement,
        s.street,
        s.house,
        s.building,
        s.block,
        s.flat,
        s.office
    from json_table(
        :sphere_json,
        '$[*]'
        columns (
            sphere_row_id number path '$.sphere_row_id',
            contract_id varchar2(200) path '$.contract_id',
            contract_number varchar2(500) path '$.contract_number',
            task_id varchar2(200) path '$.task_id',
            task_object_link_id varchar2(200)
                path '$.task_object_link_id',
            characteristics_id varchar2(200)
                path '$.characteristics_id',
            object_id varchar2(200) path '$.object_id',
            geo_address_id varchar2(200) path '$.geo_address_id',
            real_estate_objects_in_contract varchar2(200)
                path '$.real_estate_objects_in_contract',
            object_description varchar2(4000)
                path '$.object_description',
            total_area varchar2(200) path '$.total_area',
            full_address varchar2(4000) path '$.full_address',
            original_address varchar2(4000) path '$.original_address',
            postal_code varchar2(100) path '$.postal_code',
            settlement varchar2(1000) path '$.settlement',
            street varchar2(1000) path '$.street',
            house varchar2(500) path '$.house',
            building varchar2(500) path '$.building',
            block varchar2(500) path '$.block',
            flat varchar2(500) path '$.flat',
            office varchar2(500) path '$.office'
        )
    ) s
),

sphere_text as (
    /* Если нормализованного адреса нет, используем адрес, введённый вручную. */
    select
        s.*,
        replace(
            regexp_replace(trim(s.total_area), '[[:space:]]+', ''),
            ',',
            '.'
        ) as sphere_area_text,
        coalesce(
            nullif(trim(s.full_address), ''),
            nullif(trim(s.original_address), '')
        ) as source_address,
        regexp_replace(
            regexp_replace(
                replace(
                    lower(
                        replace(
                            coalesce(
                                nullif(trim(s.full_address), ''),
                                nullif(trim(s.original_address), '')
                            ),
                            chr(160),
                            ' '
                        )
                    ),
                    'ё',
                    'е'
                ),
                '[;|]+',
                ','
            ),
            '[[:space:]]*,[[:space:]]*',
            ', '
        ) as address_text
    from sphere_source s
),

sphere_parts_raw as (
    /* Берём готовые части адреса, а при их отсутствии разбираем полный адрес. */
    select
        s.*,
        coalesce(
            nullif(trim(s.postal_code), ''),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*([0-9]{6})([[:space:]]*,|$)',
                1, 1, 'i', 2
            )
        ) as postal_code_raw,
        regexp_substr(
            s.address_text,
            '(^|,)[[:space:]]*([^,]*(область|обл[.]?|край|республика|респ[.]?)[^,]*)',
            1, 1, 'i', 2
        ) as region_raw,
        coalesce(
            nullif(trim(s.settlement), ''),
            regexp_substr(
                s.address_text,
                '^[[:space:]]*([^,(]+)[[:space:]]*[(]',
                1, 1, 'i', 1
            ),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*(город|г)[.]?[[:space:]]+([^,]+)',
                1, 1, 'i', 3
            ),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*(пгт|поселок городского типа|рабочий поселок|р[.]?п|поселок|пос|село|с|деревня|д|хутор|х)[.]?[[:space:]]+([^,]+)',
                1, 1, 'i', 3
            ),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*([^,]+)[[:space:]]*,[[:space:]]*([^,]*(улица|ул[.]?|проспект|пр-кт|переулок|пер[.]?|шоссе|ш[.]?|набережная|наб[.]?|бульвар|б-р|проезд|площадь|пл[.]?|тракт|аллея))([[:space:]]*,|$)',
                1, 1, 'i', 2
            )
        ) as locality_raw,
        coalesce(
            nullif(trim(s.street), ''),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*(улица|ул[.]?|проспект|пр-кт|переулок|пер[.]?|шоссе|ш[.]?|набережная|наб[.]?|бульвар|б-р|проезд|площадь|пл[.]?|тракт|аллея)[[:space:]]+([^,]+)',
                1, 1, 'i', 3
            ),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*([^,]+)[[:space:]]+(улица|ул[.]?|проспект|пр-кт|переулок|пер[.]?|шоссе|ш[.]?|набережная|наб[.]?|бульвар|б-р|проезд|площадь|пл[.]?|тракт|аллея)([[:space:]]*,|$)',
                1, 1, 'i', 2
            )
        ) as street_raw,
        coalesce(
            nullif(trim(s.house), ''),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*(дом|д)[.]?[[:space:]]*([0-9]+[а-яa-z]?([/-][0-9а-яa-z]+)?)',
                1, 1, 'i', 3
            ),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*([0-9]{1,5}[а-яa-z]?([/-][0-9а-яa-z]+)?)([[:space:]]*,|$)',
                1, 1, 'i', 2
            )
        ) as house_raw,
        coalesce(
            nullif(trim(s.block), ''),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*(корпус|корп|к)([.]|[[:space:]])+[[:space:]]*([0-9а-яa-z/-]+)',
                1, 1, 'i', 4
            )
        ) as korpus_raw,
        coalesce(
            nullif(trim(s.building), ''),
            regexp_substr(
                s.address_text,
                '(^|,|[[:space:]])(строение|стр)[.]?[[:space:]]*([0-9а-яa-z/-]+)',
                1, 1, 'i', 3
            )
        ) as stroenie_raw
    from sphere_text s
),

sphere_prepared as (
    /* Приводим части адреса к одному виду для сравнения. */
    select /*+ materialize */
        s.*,
        case
            when regexp_like(
                s.sphere_area_text,
                '^[0-9]+([.][0-9]+)?$'
            )
            then to_number(
                s.sphere_area_text,
                '999999999999999999999999D9999999999',
                'NLS_NUMERIC_CHARACTERS=''.,'''
            )
        end as sphere_area,
        regexp_replace(s.postal_code_raw, '[^0-9]+', '')
            as sphere_postal_code,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(s.region_raw)), 'ё', 'е'),
                '(^|[[:space:]])(область|обл|край|республика|респ)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as sphere_region,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(s.locality_raw)), 'ё', 'е'),
                '(^|[[:space:]])(город|г|пгт|поселок городского типа|рабочий поселок|р[.]?п|поселок|пос|село|с|деревня|д|хутор|х)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as sphere_locality,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(s.street_raw)), 'ё', 'е'),
                '(^|[[:space:]])(улица|ул|проспект|пр-кт|переулок|пер|шоссе|ш|набережная|наб|бульвар|б-р|проезд|площадь|пл|тракт|аллея)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as sphere_street,
        regexp_replace(
            replace(lower(trim(s.house_raw)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as sphere_house,
        regexp_replace(
            replace(lower(trim(s.korpus_raw)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as sphere_korpus,
        regexp_replace(
            replace(lower(trim(s.stroenie_raw)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as sphere_stroenie
    from sphere_parts_raw s
),

sphere_core_keys as (
    /* Короткий список адресов ограничивает поиск по большой таблице ЕГРН. */
    select distinct
        sphere_locality,
        sphere_street,
        sphere_house
    from sphere_prepared
    where sphere_locality is not null
      and sphere_street is not null
      and sphere_house is not null
),

egrn_normalized as (
    /* Готовим адрес и минимальный набор данных ЕГРН. */
    select /*+ materialize */
        coalesce(
            nullif(trim(e.cadaster), ''),
            'CAD_IND:' || to_char(e.cad_ind)
        ) as egrn_key,
        e.cad_ind,
        e.cadaster,
        e.egrn_address,
        e.square,
        e.measure,
        e.building_type,
        e.oks_type,
        e.oks_purpose,
        e.object_status,
        e.fias_level,
        e.fias_id_house,
        e.row_update_date,
        e.ias_update_date,
        regexp_replace(trim(e.postal_code), '[^0-9]+', '')
            as egrn_postal_code,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(e.region)), 'ё', 'е'),
                '(^|[[:space:]])(область|обл|край|республика|респ)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as egrn_region,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(e.city)), 'ё', 'е'),
                '(^|[[:space:]])(город|г)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as egrn_city,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(e.settlement)), 'ё', 'е'),
                '(^|[[:space:]])(пгт|поселок городского типа|рабочий поселок|р[.]?п|поселок|пос|село|с|деревня|д|хутор|х)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as egrn_settlement,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(e.street)), 'ё', 'е'),
                '(^|[[:space:]])(улица|ул|проспект|пр-кт|переулок|пер|шоссе|ш|набережная|наб|бульвар|б-р|проезд|площадь|пл|тракт|аллея)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as egrn_street,
        regexp_replace(
            replace(lower(trim(e.house_number)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as egrn_house,
        regexp_replace(
            replace(lower(trim(e.vladenie)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as egrn_vladenie,
        regexp_replace(
            replace(lower(trim(e.korpus)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as egrn_korpus,
        regexp_replace(
            replace(lower(trim(e.stroenie)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as egrn_stroenie
    from DM_RISK_AVATAR.EGRN_DATA e
    where upper(trim(e.fias_level)) = 'FIAS_HOUSE'
      and lower(trim(e.oks_type)) in (
          'здание',
          'сооружение',
          'строение'
      )
      and e.flat is null
      and e.flat2 is null
      and e.office is null
      and e.office2 is null
      and e.room is null
      and e.room2 is null
      and e.compartment1 is null
      and e.compartment2 is null
      and (
          e.cadaster is not null
          or e.cad_ind is not null
      )
),

egrn_candidates as (
    /* Оставляем только адреса, которые могут относиться к нашей выгрузке. */
    select e.*
    from egrn_normalized e
    join sphere_core_keys k
        on k.sphere_street = e.egrn_street
       and k.sphere_house in (e.egrn_house, e.egrn_vladenie)
       and k.sphere_locality in (e.egrn_city, e.egrn_settlement)
),

address_matches as (
    /* Сравниваем адрес только до уровня здания. */
    select
        s.sphere_row_id,
        s.sphere_area,
        e.*
    from sphere_prepared s
    join egrn_candidates e
       on s.sphere_street = e.egrn_street
       and s.sphere_house in (e.egrn_house, e.egrn_vladenie)
       and s.sphere_locality in (e.egrn_city, e.egrn_settlement)
       and (
           s.sphere_region is null
           or e.egrn_region is null
           or s.sphere_region = e.egrn_region
       )
       and (
           s.sphere_postal_code is null
           or e.egrn_postal_code is null
           or s.sphere_postal_code = e.egrn_postal_code
       )
       and (
           s.sphere_korpus is null
           or s.sphere_korpus = e.egrn_korpus
       )
       and (
           s.sphere_stroenie is null
           or s.sphere_stroenie = e.egrn_stroenie
       )
    where s.sphere_locality is not null
      and s.sphere_street is not null
      and s.sphere_house is not null
),

ranked_egrn_rows as (
    /* Один кадастровый объект может повторяться. Оставляем свежую запись. */
    select
        m.*,
        row_number() over (
            partition by m.sphere_row_id, m.egrn_key
            order by
                m.row_update_date desc nulls last,
                m.ias_update_date desc nulls last,
                m.cad_ind desc nulls last
        ) as egrn_row_number
    from address_matches m
),

one_row_per_egrn_object as (
    select r.*
    from ranked_egrn_rows r
    where r.egrn_row_number = 1
),

area_check as (
    /* Площадь проверяем только там, где она есть с обеих сторон. */
    select
        c.*,
        count(*) over (
            partition by c.sphere_row_id
        ) as address_candidate_count,
        case
            when c.sphere_area > 0
             and c.square is not null
             and abs(c.square - c.sphere_area)
                 <= greatest(1, c.sphere_area * 0.01)
                then 1
            else 0
        end as area_matches
    from one_row_per_egrn_object c
),

area_choice as (
    select
        a.*,
        max(a.area_matches) over (
            partition by a.sphere_row_id
        ) as has_area_match
    from area_check a
),

candidates_after_area as (
    /* Если площадь помогла, оставляем совпавших. Иначе никого не отсекаем. */
    select a.*
    from area_choice a
    where a.has_area_match = 0
       or a.area_matches = 1
),

candidate_counts as (
    select
        c.*,
        count(*) over (
            partition by c.sphere_row_id
        ) as candidate_count
    from candidates_after_area c
),

candidate_summary as (
    select
        c.sphere_row_id,
        max(c.candidate_count) as candidate_count,
        max(c.address_candidate_count) as address_candidate_count,
        max(c.has_area_match) as has_area_match
    from candidate_counts c
    group by c.sphere_row_id
),

chosen_egrn as (
    /* Здание ЕГРН присоединяется только при одном кандидате. */
    select c.*
    from candidate_counts c
    where c.candidate_count = 1
)

select
    sphere.sphere_row_id as "sphere_row_id",

    /* Договор и основные ID строки Сферы. */
    sphere.contract_number as "Номер договора",
    sphere.contract_id as "ID договора",
    sphere.task_id as "ID задачи",
    sphere.task_object_link_id as "ID связи задачи и объекта",
    sphere.characteristics_id as "ID характеристик",
    sphere.object_id as "ID объекта Сферы",
    sphere.geo_address_id as "ID адреса Сферы",
    sphere.real_estate_objects_in_contract
        as "Объектов недвижимости в договоре",

    /* Объект и исходные адресные строки. */
    sphere.object_description as "Описание объекта Сферы",
    sphere.full_address as "Полный адрес Сферы",
    sphere.original_address as "Исходный адрес Сферы",
    prepared.source_address as "Адрес, который разбирал запрос",
    sphere.total_area as "Площадь Сферы",

    /* Части адреса, которые уже лежали в отдельных колонках Сферы. */
    sphere.postal_code as "Сфера: почтовый индекс",
    sphere.settlement as "Сфера: населённый пункт",
    sphere.street as "Сфера: улица",
    sphere.house as "Сфера: дом",
    sphere.block as "Сфера: корпус",
    sphere.building as "Сфера: строение",
    sphere.flat as "Сфера: квартира или помещение",
    sphere.office as "Сфера: офис",

    /* Так запрос разбил исходную строку адреса до очистки. */
    prepared.postal_code_raw as "После разбора: почтовый индекс",
    prepared.region_raw as "После разбора: регион",
    prepared.locality_raw as "После разбора: населённый пункт",
    prepared.street_raw as "После разбора: улица",
    prepared.house_raw as "После разбора: дом",
    prepared.korpus_raw as "После разбора: корпус",
    prepared.stroenie_raw as "После разбора: строение",

    /* Итог поиска. */
    case
        when prepared.source_address is null
            then 'В Сфере нет адреса'
        when prepared.sphere_locality is null
          or prepared.sphere_street is null
          or prepared.sphere_house is null
            then 'Не удалось выделить населённый пункт, улицу или дом'
        when nvl(summary.candidate_count, 0) = 0
            then 'Здание ЕГРН не найдено'
        when summary.candidate_count = 1
            then 'Найдено одно здание ЕГРН'
        else 'Найдено несколько зданий. ЕГРН не присоединён'
    end as "Результат поиска",
    nvl(summary.address_candidate_count, 0)
        as "Кандидатов по адресу",
    nvl(summary.candidate_count, 0)
        as "Кандидатов после площади",
    case
        when summary.candidate_count = 1 then 1
        else 0
    end as "Соединение ЕГРН единичное",
    case
        when prepared.source_address is null
            then 'Нет адреса в Сфере'
        when prepared.sphere_locality is null
          or prepared.sphere_street is null
          or prepared.sphere_house is null
            then 'Адрес не удалось разобрать до здания'
        when nvl(summary.candidate_count, 0) = 0
            then 'Совпадение ЕГРН не найдено'
        when summary.candidate_count > 1
            then 'Неоднозначное совпадение'
        when summary.has_area_match = 1
            then 'Адрес здания и площадь'
        when prepared.sphere_area is null
            then 'Только адрес здания: в Сфере нет площади'
        when chosen.square is null
            then 'Только адрес здания: в ЕГРН нет площади'
        else 'Только адрес здания: площадь не подтвердила совпадение'
    end as "Основание соединения ЕГРН",
    case
        when summary.address_candidate_count > summary.candidate_count
            then 'Да'
        else 'Нет'
    end as "Площадь помогла сузить поиск",

    /* Очищенные значения показаны парами: Сфера и найденное здание КХД. */
    prepared.sphere_postal_code as "Сравнение: индекс Сферы",
    chosen.egrn_postal_code as "Сравнение: индекс КХД",

    prepared.sphere_region as "Сравнение: регион Сферы",
    chosen.egrn_region as "Сравнение: регион КХД",

    prepared.sphere_locality as "Сравнение: населённый пункт Сферы",
    case
        when prepared.sphere_locality = chosen.egrn_city
            then chosen.egrn_city
        when prepared.sphere_locality = chosen.egrn_settlement
            then chosen.egrn_settlement
    end as "Сравнение: населённый пункт КХД",

    prepared.sphere_street as "Сравнение: улица Сферы",
    chosen.egrn_street as "Сравнение: улица КХД",

    prepared.sphere_house as "Сравнение: дом Сферы",
    case
        when prepared.sphere_house = chosen.egrn_house
            then chosen.egrn_house
        when prepared.sphere_house = chosen.egrn_vladenie
            then chosen.egrn_vladenie
    end as "Сравнение: дом КХД",

    prepared.sphere_korpus as "Сравнение: корпус Сферы",
    chosen.egrn_korpus as "Сравнение: корпус КХД",

    prepared.sphere_stroenie as "Сравнение: строение Сферы",
    chosen.egrn_stroenie as "Сравнение: строение КХД",

    /* Поля одного найденного здания. */
    chosen.cad_ind as "Внутренний ID здания ЕГРН",
    chosen.cadaster as "Кадастровый номер здания",
    chosen.egrn_address as "Адрес здания ЕГРН",
    chosen.square as "Площадь здания ЕГРН",
    chosen.measure as "Единица площади здания",
    chosen.building_type as "Тип строения здания ЕГРН",
    chosen.oks_type as "Тип здания ЕГРН",
    chosen.oks_purpose as "Назначение здания ЕГРН",
    chosen.object_status as "Статус здания ЕГРН",
    chosen.fias_level as "Уровень адреса ЕГРН",
    chosen.fias_id_house as "ФИАС дома ЕГРН",
    case
        when chosen.cad_ind is not null then 'Здание'
    end as "Уровень присоединения"

from sphere_source sphere
join sphere_prepared prepared
    on prepared.sphere_row_id = sphere.sphere_row_id
left join candidate_summary summary
    on summary.sphere_row_id = sphere.sphere_row_id
left join chosen_egrn chosen
    on chosen.sphere_row_id = sphere.sphere_row_id
order by
    sphere.contract_number,
    sphere.object_id

/*
Как читать результат
--------------------
Найдено одно здание ЕГРН
    Здание присоединено ко всем объектам Сферы с этим адресом.

Найдено несколько зданий
    По адресу есть несколько кадастровых зданий. Ничего не присоединено.

Здание ЕГРН не найдено
    Адрес удалось разобрать, но запись уровня FIAS_HOUSE не найдена.

Не удалось выделить населённый пункт, улицу или дом
    Адрес есть, но его недостаточно для безопасного автоматического поиска.
*/

"""


In [ ]:
# выполняем Oracle SQL и получаем одну строку результата на строку Сферы
if not KHD_DATA_SCHEMA.replace('_', '').isalnum():
    raise ValueError('Некорректное имя схемы КХД')

egrn_query = egrn_sql.replace(
    'DM_RISK_AVATAR.EGRN_DATA',
    f'{KHD_DATA_SCHEMA.upper()}.EGRN_DATA',
)
with khd_connection.cursor() as cursor:
    sphere_json_bind = cursor.var(oracledb.DB_TYPE_CLOB)
    sphere_json_bind.setvalue(0, sphere_json)
    cursor.execute(egrn_query, sphere_json=sphere_json_bind)
    egrn_result_columns = [column[0] for column in cursor.description]
    egrn_result_rows = cursor.fetchall()

egrn_lookup_df = pd.DataFrame(
    egrn_result_rows,
    columns=egrn_result_columns,
)
if 'SPHERE_ROW_ID' in egrn_lookup_df.columns:
    egrn_lookup_df = egrn_lookup_df.rename(
        columns={'SPHERE_ROW_ID': 'sphere_row_id'}
    )

egrn_column_names = {
    'Результат поиска': 'egrn_match_status',
    'Кандидатов по адресу': 'egrn_address_candidate_count',
    'Кандидатов после площади': 'egrn_candidate_count',
    'Соединение ЕГРН единичное': 'egrn_is_unique_match',
    'Основание соединения ЕГРН': 'egrn_match_basis',
    'Площадь помогла сузить поиск': 'egrn_area_narrowed_search',
    'Адрес, который разбирал запрос': 'sphere_address_for_egrn',
    'После разбора: почтовый индекс': 'parsed_postal_code',
    'После разбора: регион': 'parsed_region',
    'После разбора: населённый пункт': 'parsed_locality',
    'После разбора: улица': 'parsed_street',
    'После разбора: дом': 'parsed_house',
    'После разбора: корпус': 'parsed_korpus',
    'После разбора: строение': 'parsed_stroenie',
    'Сравнение: индекс Сферы': 'compared_sphere_postal_code',
    'Сравнение: индекс КХД': 'matched_egrn_postal_code',
    'Сравнение: регион Сферы': 'compared_sphere_region',
    'Сравнение: регион КХД': 'matched_egrn_region',
    'Сравнение: населённый пункт Сферы': 'compared_sphere_locality',
    'Сравнение: населённый пункт КХД': 'matched_egrn_locality',
    'Сравнение: улица Сферы': 'compared_sphere_street',
    'Сравнение: улица КХД': 'matched_egrn_street',
    'Сравнение: дом Сферы': 'compared_sphere_house',
    'Сравнение: дом КХД': 'matched_egrn_house',
    'Сравнение: корпус Сферы': 'compared_sphere_korpus',
    'Сравнение: корпус КХД': 'matched_egrn_korpus',
    'Сравнение: строение Сферы': 'compared_sphere_stroenie',
    'Сравнение: строение КХД': 'matched_egrn_stroenie',
    'Внутренний ID здания ЕГРН': 'egrn_cad_ind',
    'Кадастровый номер здания': 'egrn_cadaster',
    'Адрес здания ЕГРН': 'egrn_address',
    'Площадь здания ЕГРН': 'egrn_square',
    'Единица площади здания': 'egrn_measure',
    'Тип строения здания ЕГРН': 'egrn_building_type',
    'Тип здания ЕГРН': 'egrn_oks_type',
    'Назначение здания ЕГРН': 'egrn_oks_purpose',
    'Статус здания ЕГРН': 'egrn_object_status',
    'Уровень адреса ЕГРН': 'egrn_fias_level',
    'ФИАС дома ЕГРН': 'egrn_fias_id_house',
    'Уровень присоединения': 'egrn_join_level',
}
egrn_lookup_df = egrn_lookup_df.rename(columns=egrn_column_names)
egrn_columns = ['sphere_row_id'] + list(egrn_column_names.values())

missing_egrn_columns = [
    column for column in egrn_columns
    if column not in egrn_lookup_df.columns
]
if missing_egrn_columns:
    raise ValueError(
        'Oracle не вернул ожидаемые колонки: '
        + ', '.join(missing_egrn_columns)
    )
if egrn_lookup_df['sphere_row_id'].duplicated().any():
    raise ValueError('Oracle вернул несколько строк для одного объекта Сферы')

strict_egrn_df = sphere_with_row_id.merge(
    egrn_lookup_df[egrn_columns],
    on='sphere_row_id',
    how='left',
    validate='one_to_one',
)

if len(strict_egrn_df) != len(sphere_with_row_id):
    raise ValueError('После соединения с ЕГРН изменилось количество строк')

print('Строк после соединения с ЕГРН:', len(strict_egrn_df))


# 6. Проверка соединения с ЕГРН


In [ ]:
egrn_profile = (
    strict_egrn_df['egrn_match_status']
    .fillna('Нет результата Oracle')
    .value_counts(dropna=False)
    .rename_axis('Результат поиска')
    .reset_index(name='Количество строк')
)
display(egrn_profile)

egrn_preview_columns = [
    'contract_id',
    'object_id',
    'full_address',
    'total_area',
    'egrn_match_status',
    'egrn_is_unique_match',
    'egrn_match_basis',
    'egrn_candidate_count',
    'egrn_cadaster',
    'egrn_address',
    'egrn_square',
]
display(strict_egrn_df[egrn_preview_columns].head(20))


# 7. Сохранение результата


In [ ]:
output_path = OUTPUT_DIR / 'датасет_строгий_с_егрн.csv'
strict_egrn_df.to_csv(output_path, index=False, sep=';', encoding='utf-8-sig')
print('Сохранено:', output_path)


# 8. Сохранение списка ИНН

Отдельный файл содержит только одну колонку `inn`. Пустые значения и повторы исключаются.


In [ ]:
inn_df = (
    strict_egrn_df[['policyholder_inn']]
    .rename(columns={'policyholder_inn': 'inn'})
    .assign(inn=lambda data: data['inn'].astype('string').str.strip())
    .loc[lambda data: data['inn'].notna() & data['inn'].ne('')]
    .drop_duplicates(subset=['inn'])
    .sort_values('inn')
    .reset_index(drop=True)
)

inn_path = OUTPUT_DIR / 'inn.csv'
inn_df.to_csv(inn_path, index=False, sep=';', encoding='utf-8-sig')
print('Уникальных ИНН:', len(inn_df))
print('Сохранено:', inn_path)


In [ ]:
engine.dispose()
khd_connection.close()
print('Подключения к Сфере и КХД закрыты')